# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ibraheem-ali23/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

Unit of analysis: One row represents a single anonymized client web page (within the Content Refresh & Discoverability lane) evaluated over a rolling/defined observation and outcome window (iterating on mid-panel month e.g. 2026-03 while keeping the final month sealed for evaluation).

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, sys, subprocess
import pandas as pd
import numpy as np

if "google.colab" in sys.modules and not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    if not os.path.isdir("flyrank-ml-internship"):
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/ibraheem-ali23/flyrank-ml-internship.git"],
            check=True,
        )
    os.chdir("flyrank-ml-internship")
elif not os.path.exists("data/raw/content_refresh_anonymized.csv") and os.path.exists("../../data/raw/content_refresh_anonymized.csv"):
    os.chdir("../..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Shape:", df.shape)
print("Columns:", list(df.columns))

Shape: (30000, 44)
Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## 2. Fields: feature / label / context / excluded

- **Features (knowable at decision time):** Pre-decision content freshness (`days_since_last_update`, `content_age_days`), visibility & engagement (`impressions_90d`, `avg_position`, `ctr`), and content scale (`word_count`) — max 5-6 core features, each observable before taking refresh action.
- **Label / Target proxy:** Binary indicator of traffic decline (`is_declining_label` derived from `trend_direction == 'down'`), used as a directional proxy for content decay.
- **Context / Grouping:** Anonymized page/client identifiers and observation slice metadata for grouping and holdout evaluation (no raw client names, URLs, or private queries).
- **Excluded (and why):** Direct outcome/post-decision metrics such as `trend_pct` or future traffic deltas are excluded because including label-derived signals causes target leakage and inflates performance artificially.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df["is_declining_label"] = df["trend_direction"].astype(str).str.lower().eq("down").astype(int)

candidate_features = [
    c for c in ["content_age_days", "days_since_last_update", "impressions_90d", "avg_position", "ctr", "word_count"]
    if c in df.columns
]

# Deliberate leakage trap: adding a label-derived column inflates signal artificially, then we drop it
df["leaky_label_copy"] = df["is_declining_label"]
leaky_corr = df[["leaky_label_copy", "is_declining_label"]].corr().iloc[0, 1]
print(f"Leakage trap correlation (must remove): {leaky_corr:.4f}")

# Drop leaky column and keep clean pre-decision feature set
df = df.drop(columns=["leaky_label_copy"])
print("Clean feature columns:", candidate_features)


Leakage trap correlation (must remove): 1.0000
Clean feature columns: ['content_age_days', 'days_since_last_update', 'impressions_90d', 'avg_position', 'ctr', 'word_count']


## 3. Verify it with queries (grain, counts, missing values, windows)

Every contract claim is verified programmatically below: checking row count and grain, missingness per column, and target/feature availability across valid non-null rows.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Query 1: Grain & row count check
total_rows = len(df)
print(f"Query 1 - Total rows: {total_rows:,}")

# Query 2: Missing values & completeness check
missing_summary = df[candidate_features + ["is_declining_label"]].isna().sum()
print("\nQuery 2 - Missing values per column:")
print(missing_summary)

# Query 3: Availability / valid non-null rows surviving check
valid_mask = df[candidate_features + ["is_declining_label"]].notna().all(axis=1)
print(f"\nQuery 3 - Valid complete rows: {valid_mask.sum():,} / {total_rows:,} ({valid_mask.mean()*100:.1f}%)")
print(f"Declining share in valid slice: {df.loc[valid_mask, 'is_declining_label'].mean():.3f}")

Query 1 - Total rows: 30,000

Query 2 - Missing values per column:
content_age_days             0
days_since_last_update       0
impressions_90d              0
avg_position                 0
ctr                          0
word_count                7699
is_declining_label           0
dtype: int64

Query 3 - Valid complete rows: 22,301 / 30,000 (74.3%)
Declining share in valid slice: 0.569


## 4. Data limits

Data limitations: Observed historical search and content metrics provide directional decision-support signals, not causal proof or a guarantee of search engine ranking behavior. Rolling/overlapping windows, potential history imbalance across pages or clients, and proxy label definitions mean results should be validated carefully on out-of-time / holdout slices before acting.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
summary_stats = df[candidate_features].describe().T[["count", "mean", "std", "min", "50%", "max"]]
summary_stats


,count,mean,std,min,50%,max
content_age_days,30000.0,256.167800,132.707930,90.0,236.00,564.0
days_since_last_update,30000.0,46.098300,42.078709,1.0,20.00,373.0
impressions_90d,30000.0,5200.366300,16838.019547,1.0,731.00,517715.0
avg_position,30000.0,16.342380,15.216790,0.0,10.80,245.0
ctr,30000.0,0.510733,3.279162,0.0,0.07,100.0
word_count,22301.0,3107.760325,1452.382598,8.0,2877.00,9546.0


## Self-check

Before you submit, confirm each line honestly:

- [X ] Every section above is filled — markdown thinking AND the code that backs it
- [ X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X ] No client names, URLs, or private queries anywhere
- [ X] My claims use careful words: observed, measured, directional, decision-support
- [ X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.